# Laboratório 04 — Regressão linear e gradiente descendente

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Nos três laboratórios anteriores você implementou **busca**. Hoje muda o assunto:
pela primeira vez o programa vai **aprender** alguma coisa — e aprender, aqui, quer
dizer *ajustar dois números até que um erro fique pequeno*.

Você vai escrever, do zero, as quatro funções que formam o algoritmo:

| Função | O que faz |
|---|---|
| `prever` | aplica o modelo $\hat y = b_0 + b_1 x$ |
| `custo` | mede o erro quadrático médio |
| `gradientes` | calcula as duas derivadas |
| `gradiente_descendente` | o laço que junta tudo |

E só **depois** de tudo funcionar é que a biblioteca entra, para comparar.

> ⚙️ Só `numpy` e `matplotlib` até a Parte 8. O `scikit-learn` aparece no fim,
> de propósito.

> 💻 **Roda em qualquer lugar.** O notebook não lê arquivo nenhum: os dados são
> gerados com semente fixa. Funciona no laboratório, em casa e no Google Colab.

> 🎯 Os dados são os **mesmos dos slides**. Se tudo der certo, você vai reencontrar
> $b_0 = 52{,}5181$ e $b_1 = 3{,}2901$.


## Parte 0 — Os dados

Quarenta apartamentos de Itumbiara: **área** em $m^2$ e **preço** em mil R\$.

São dados sintéticos, gerados com semente fixa — então todo mundo da turma tem
exatamente os mesmos quarenta imóveis, e os mesmos resultados.

Rode a célula e olhe o gráfico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEMENTE = 20260922
rng = np.random.default_rng(SEMENTE)

n = 40
area  = np.round(rng.uniform(45, 155, n), 0)                   # m2
preco = np.round(62.0 + 3.20 * area + rng.normal(0, 22, n), 1)  # mil R$

print(f"{n} imóveis")
print(f"área  : de {area.min():.0f} a {area.max():.0f} m2  (média {area.mean():.1f})")
print(f"preço : de {preco.min():.1f} a {preco.max():.1f} mil R$  (média {preco.mean():.1f})")

plt.figure(figsize=(6.5, 4))
plt.scatter(area, preco, s=28, color="#0F5028", alpha=0.75)
plt.xlabel("área ($m^2$)")
plt.ylabel("preço (mil R$)")
plt.title("Os 40 imóveis")
plt.grid(alpha=0.3)
plt.show()

## Parte 1 — O modelo e o custo

O modelo tem uma linha só:

$$\hat y = b_0 + b_1 x$$

E o custo é o **erro quadrático médio**:

$$\mathrm{MSE}(b_0, b_1) = \frac{1}{n}\sum_{i=1}^{n}(\hat y_i - y_i)^2$$

Repare nos argumentos: o custo é função **dos parâmetros**, não dos dados. Os dados
estão fixos; quem se mexe são $b_0$ e $b_1$.

> 💡 Use operações vetorizadas do NumPy. `b0 + b1 * x` já devolve o vetor inteiro
> de previsões — não precisa de laço.

In [ ]:
def prever(x, b0, b1):
    """Devolve o vetor de previsões do modelo."""
    # TODO 1: aplique o modelo. Uma linha.
    return np.zeros_like(x)


def custo(x, y, b0, b1):
    """Erro quadrático médio das previsões em relação a y."""
    # TODO 2: calcule os resíduos (previsto - real), eleve ao quadrado
    #         e tire a média.  Dica: np.mean(...)
    return 0.0

**Confira.** A melhor reta possível para estes dados é
$\hat y = 52{,}52 + 3{,}29x$, e o MSE dela é **306,07**. As outras duas retas são
piores — é o gráfico das três retas do slide.

In [ ]:
for b0, b1, rotulo in [(0.0, 4.00, "só inclinação"),
                       (150.0, 2.00, "começa alto"),
                       (52.52, 3.29, "a melhor")]:
    c = custo(area, preco, b0, b1)
    print(f"b0={b0:7.2f}  b1={b1:4.2f}   MSE = {c:9.2f}   "
          f"RMSE = R$ {np.sqrt(c):6.2f} mil   ({rotulo})")

## Parte 2 — Os gradientes

As duas derivadas parciais do MSE:

$$\frac{\partial\,\mathrm{MSE}}{\partial b_0} = \frac{2}{n}\sum(\hat y_i - y_i)
\qquad
\frac{\partial\,\mathrm{MSE}}{\partial b_1} = \frac{2}{n}\sum(\hat y_i - y_i)\,x_i$$

São a mesma soma; a diferença é o $x_i$ multiplicando na segunda.

In [ ]:
def gradientes(x, y, b0, b1):
    """Devolve a tupla (g0, g1) com as duas derivadas parciais."""
    n = len(x)
    residuos = prever(x, b0, b1) - y
    # TODO 3: monte g0 e g1 pelas fórmulas acima.
    #         Dica: np.sum(residuos) e np.sum(residuos * x)
    g0 = 0.0
    g1 = 0.0
    return g0, g1

**Confira com o exemplo do slide.** Com $x = (1,2,3,4)$, $y = (3,5,7,10)$ e
$b_0 = b_1 = 0$, os gradientes têm que dar exatamente **-12,5** e **-37**, e o MSE
inicial **45,75**.

In [ ]:
x_mao = np.array([1.0, 2.0, 3.0, 4.0])
y_mao = np.array([3.0, 5.0, 7.0, 10.0])

print("MSE inicial:", custo(x_mao, y_mao, 0.0, 0.0), "   (esperado: 45.75)")
print("gradientes :", gradientes(x_mao, y_mao, 0.0, 0.0), "   (esperado: -12.5, -37.0)")

## Parte 3 — O laço

O algoritmo inteiro, em cinco passos:

1. comece com um chute ($b_0 = b_1 = 0$ serve);
2. calcule os gradientes;
3. ande **contra** eles: $b \leftarrow b - \eta\,g$;
4. guarde o custo, para poder desenhar a curva depois;
5. repita.

> ⚠️ **Atualize os dois parâmetros com os gradientes velhos.** Se você atualizar
> `b0` e só então chamar `gradientes` de novo para o `b1`, o algoritmo ainda roda e
> ainda converge — só que mais devagar e por um caminho errado. É um bug que não dá
> sintoma nenhum, e por isso é o mais comum aqui.

In [ ]:
def gradiente_descendente(x, y, eta, passos, b0=0.0, b1=0.0):
    """Devolve (b0, b1, historia_do_custo)."""
    historia = []
    for _ in range(passos):
        historia.append(custo(x, y, b0, b1))
        # TODO 4: calcule os dois gradientes DE UMA VEZ e atualize os dois
        #         parâmetros. Não recalcule o gradiente entre uma atualização
        #         e outra.
        pass
    historia.append(custo(x, y, b0, b1))
    return b0, b1, historia

**Confira com o exemplo do slide.** Um passo só, com $\eta = 0{,}1$: os
parâmetros têm que virar **1,25** e **3,70**, e o custo cair de 45,75 para **20,59**.

E rodando até o fim, a resposta exata é $b_0 = 0{,}5$ e $b_1 = 2{,}3$.

In [ ]:
b0, b1, hist = gradiente_descendente(x_mao, y_mao, eta=0.1, passos=1)
print(f"1 passo   : b0 = {b0:.4f}  b1 = {b1:.4f}   custo = {hist[-1]:.4f}")
print( "            (esperado: 1.2500, 3.7000, 20.5875)")

b0, b1, hist = gradiente_descendente(x_mao, y_mao, eta=0.1, passos=2000)
print(f"2000 passos: b0 = {b0:.4f}  b1 = {b1:.4f}   custo = {hist[-1]:.4f}")
print( "            (esperado: 0.5000, 2.3000, 0.0750)")

## Parte 4 — Agora nos dados de verdade

Funcionou nos quatro pontinhos. Vamos aos quarenta apartamentos.

Rode e observe **antes de ler a próxima célula**.

In [ ]:
# O np.errstate silencia os avisos de "overflow" do NumPy. Eles apareceriam
# nas duas últimas linhas, e são exatamente o sintoma que queremos observar:
# os números crescem tanto que estouram o ponto flutuante.
with np.errstate(over="ignore", invalid="ignore"):
    for eta in [1e-6, 1e-5, 1e-4, 1e-3]:
        b0, b1, hist = gradiente_descendente(area, preco, eta=eta, passos=1000)
        final = hist[-1]
        estado = ("DIVERGIU" if (not np.isfinite(final) or final > 1e6)
                  else f"MSE = {final:9.2f}")
        print(f"eta = {eta:.0e}   ->   {estado}")

print("\nO menor MSE possível nestes dados é 306,07.")

**O que aconteceu.** Abaixo de $10^{-4}$ o método rasteja e estaciona longe do
mínimo; em $10^{-4}$ já explode. **Não existe taxa boa** — a janela entre "lento
demais" e "instável" simplesmente não existe nestes dados.

O motivo é a **escala**. A área vai até 155, e o gradiente de $b_1$ é multiplicado
por $x_i$ — então ele fica cerca de cem vezes maior que o de $b_0$. Um passo que
serve para um destrói o outro.

## Parte 5 — Padronizar

A saída é trocar cada valor por *quantos desvios-padrão da média ele está*:

$$z_i = \frac{x_i - \mu}{\sigma}$$

Isso não muda o problema: muda o **formato do vale** que o gradiente percorre.

In [ ]:
def padronizar(x):
    """Devolve (z, mu, sigma)."""
    # TODO 5: calcule a média e o desvio-padrão de x e devolva os três valores.
    #         Dica: x.mean() e x.std()
    return x, 0.0, 1.0


z, mu, sigma = padronizar(area)
print(f"mu = {mu:.4f}   sigma = {sigma:.4f}")
print(f"z: média {z.mean():.6f} (deve dar 0)   desvio {z.std():.6f} (deve dar 1)")

Agora o mesmo laço, nos dados padronizados. Compare com a Parte 4: lá nenhuma
taxa servia; aqui a faixa que funciona é larga.

In [ ]:
for eta in [0.001, 0.01, 0.1, 0.5, 1.1]:
    b0, b1, hist = gradiente_descendente(z, preco, eta=eta, passos=1000)
    final = hist[-1]
    estado = "DIVERGIU" if (not np.isfinite(final) or final > 1e6) else f"MSE = {final:9.2f}"
    print(f"eta = {eta:<6} ->   {estado}")

## Parte 6 — A curva de custo

**Este é o gráfico que você vai olhar pelo resto da vida ao treinar qualquer
modelo.** Três formas, três diagnósticos:

- desce quase nada → taxa pequena demais;
- desce e estabiliza → certo;
- **sobe** → taxa grande demais, sempre.

In [ ]:
plt.figure(figsize=(6.5, 4))
for eta, cor, estilo in [(0.001, "#7A7A7A", "--"),
                         (0.1,   "#0F5028", "-"),
                         (1.1,   "#B02020", "-.")]:
    _, _, hist = gradiente_descendente(z, preco, eta=eta, passos=60)
    h = [min(v, 1e6) if np.isfinite(v) else 1e6 for v in hist]
    plt.plot(h, estilo, color=cor, lw=2, label=f"eta = {eta}")

plt.axhline(306.07, color="#C8641E", ls=":", lw=1.6, label="mínimo possível")
plt.yscale("log")
plt.xlabel("iteração")
plt.ylabel("MSE (escala log)")
plt.title("A taxa de aprendizado decide tudo")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

## Parte 7 — Desfazer a padronização

Os coeficientes que saíram do treino estão na escala padronizada. Para explicar o
modelo a alguém, eles precisam voltar:

$$b_1 = \frac{b_1^{z}}{\sigma}
\qquad
b_0 = b_0^{z} - \frac{b_1^{z}\,\mu}{\sigma}$$

In [ ]:
b0z, b1z, _ = gradiente_descendente(z, preco, eta=0.1, passos=1000)

# TODO 6: converta (b0z, b1z) de volta para a escala original.
b1_final = 0.0
b0_final = 0.0

print(f"no espaço padronizado: b0 = {b0z:.4f}   b1 = {b1z:.4f}")
print(f"na escala original   : b0 = {b0_final:.4f}   b1 = {b1_final:.4f}")
print( "esperado             : b0 =  52.5181   b1 = 3.2901")

E a leitura no domínio: cada $m^2$ a mais vale, em média, **R\$ 3.290** no preço
previsto.

Rode a célula abaixo para ver a reta em cima dos dados.

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.scatter(area, preco, s=28, color="#0F5028", alpha=0.7, label="os 40 imóveis")
xs = np.linspace(40, 160, 2)
plt.plot(xs, prever(xs, b0_final, b1_final), color="#B02020", lw=2,
         label=f"ajuste: {b0_final:.2f} + {b1_final:.2f} x")
plt.xlabel("área ($m^2$)")
plt.ylabel("preço (mil R$)")
plt.title("A reta que o gradiente descendente encontrou")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Um apartamento de 95 m2: R$ {prever(95.0, b0_final, b1_final):.1f} mil")

## Parte 8 — A resposta exata, e a biblioteca

A regressão linear é um caso raro: ela **tem fórmula fechada**.

$$b_1 = \frac{\sum (x_i - \bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2}
\qquad
b_0 = \bar y - b_1 \bar x$$

Então dá para conferir se o método iterativo chegou onde deveria. **Este é o único
laboratório do semestre em que isso é possível** — de uma rede neural não existe
"resposta exata" com que comparar.

In [ ]:
def solucao_fechada(x, y):
    """Devolve (b0, b1) pela fórmula dos mínimos quadrados."""
    # TODO 7: implemente as duas fórmulas acima.
    #         Dica: x.mean(), y.mean(), np.sum(...)
    return 0.0, 0.0


b0_exato, b1_exato = solucao_fechada(area, preco)
print(f"solução fechada       : b0 = {b0_exato:.6f}   b1 = {b1_exato:.6f}")
print(f"gradiente descendente : b0 = {b0_final:.6f}   b1 = {b1_final:.6f}")
print(f"diferença             : {abs(b0_exato-b0_final):.2e}  e  {abs(b1_exato-b1_final):.2e}")

Agora o Scikit-Learn. **Três caminhos, o mesmo número** — a biblioteca não faz
mágica, faz a mesma conta.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression().fit(area.reshape(-1, 1), preco)
print(f"scikit-learn          : b0 = {modelo.intercept_:.6f}   b1 = {modelo.coef_[0]:.6f}")
print(f"nosso, do zero        : b0 = {b0_final:.6f}   b1 = {b1_final:.6f}")

## Parte 9 — Treino, teste e o vazamento que quase todo mundo comete

Até aqui treinamos e avaliamos no **mesmo** conjunto — o que a aula de 15/09 proibiu.
Vamos separar.

E aqui mora a armadilha: **$\mu$ e $\sigma$ da padronização têm que sair só do
treino.** Calcular a média usando também o teste é vazamento de dados, e o resultado
fica bom demais para ser verdade.

É exatamente por isso que o `StandardScaler` tem `fit_transform` (no treino) e só
`transform` (no teste).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = area.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X, preco, test_size=0.25, random_state=SEMENTE)

escala = StandardScaler()
X_tr_z = escala.fit_transform(X_tr)   # fit SÓ no treino
X_te_z = escala.transform(X_te)       # no teste, só transform

m = LinearRegression().fit(X_tr_z, y_tr)
pred = m.predict(X_te_z)

print(f"treino: {len(y_tr)} imóveis    teste: {len(y_te)} imóveis")
print(f"RMSE no teste : R$ {np.sqrt(mean_squared_error(y_te, pred)):.2f} mil")
print(f"MAE  no teste : R$ {mean_absolute_error(y_te, pred):.2f} mil")
print(f"R2   no teste : {r2_score(y_te, pred):.4f}")

**Leia os três números.** O RMSE está na unidade do problema — é o erro típico,
em mil reais, de uma previsão em imóveis que o modelo nunca viu. O $R^2$ compara o
modelo com o modelo mais burro possível, que é responder sempre a média.

## Parte 10 — Desafio (opcional)

Sem dica e sem TODO. Escolha um:

**(a)** Descubra, por tentativa, a **maior** taxa de aprendizado que ainda converge
nos dados padronizados. Comece em 1,0 e vá subindo de 0,01 em 0,01.

**(b)** Modifique o `gradiente_descendente` para **parar sozinho** quando a queda do
custo de um passo para o outro ficar menor que $10^{-6}$, e conte quantos passos ele
usou com $\eta = 0{,}1$.

**(c)** Implemente o **MAE** como função de custo no lugar do MSE e veja o que muda
no ajuste. (O gradiente é outro: a derivada do módulo é o sinal do resíduo.)

---

## Antes de sair do laboratório

As máquinas do laboratório são restauradas ao reiniciar. **Salve o notebook na
nuvem** (OneDrive institucional, Google Drive) ou envie para você mesmo — o arquivo
não sobrevive ao próximo boot.

E apague qualquer login que tenha feito no navegador.

---

## O que este laboratório mostrou

1. **Treinar é achar números.** Foram dois, e o método é o mesmo para bilhões.
2. A **função de custo** transforma "qual reta é melhor" numa conta.
3. O **gradiente descendente** anda contra a inclinação, e freia sozinho perto do
   mínimo.
4. A **taxa de aprendizado** é escolha sua. Custo subindo é sempre taxa grande
   demais.
5. **Padronizar não é capricho** — sem isso, nestes dados, nenhuma taxa funcionava.
6. $\mu$ e $\sigma$ saem **só do treino**. Vazamento de dados, de novo.
7. A biblioteca chega ao mesmo número que você. Ela não faz mágica: faz a mesma
   conta, mais rápido.